In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split

/home/daan/OHL-AI-Week/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
GK_DATA = Path("GK_Data")
def load_keeper_match_kpis(player_id, match_dirs_str):
    """Load player_kpis for a keeper from all his matches."""
    all_kpis = []
    if not isinstance(match_dirs_str, str) or not match_dirs_str:
        return all_kpis

    for match_dir in match_dirs_str.split("|"):
        match_dir = match_dir.strip()
        if not match_dir:
            continue
        for cs_dir in (GK_DATA / "competitions").iterdir():
            if not cs_dir.is_dir():
                continue
            pkpi_path = cs_dir / match_dir / "player_kpis.json"
            if pkpi_path.exists():
                with open(pkpi_path) as f:
                    data = json.load(f).get("data", {})
                for side in ["squadHome", "squadAway"]:
                    for player in data.get(side, {}).get("players", []):
                        if (
                            player["id"] == player_id
                            and player.get("position") == "GOALKEEPER"
                            and player.get("matchShare", 0) >= 0.5
                        ):
                            kpis = {k["kpiId"]: k["value"] for k in player.get("kpis", [])}
                            kpis["matchShare"] = player["matchShare"]
                            kpis["playDuration"] = player.get("playDuration", 0)
                            all_kpis.append(kpis)
                break  
    return all_kpis

dataset = pd.read_csv(GK_DATA / "gk_dataset_final.csv")

with open(GK_DATA / "player_kpi_definitions.json") as f:
    kpi_defs = {d["id"]: d["name"] for d in json.load(f).get("data", [])}

all_match_rows = []

for _, row in dataset.iterrows():
    match_kpis = load_keeper_match_kpis(row["playerId"], row["origin_match_dirs"])
    
    if match_kpis:
        for match_data in match_kpis:
            entry = row.drop(['origin_match_dirs', 'current_match_dirs']).to_dict()
            
            for kpi_id, value in match_data.items():
                kpi_name = kpi_defs.get(kpi_id, f"KPI_{kpi_id}")
                entry[kpi_name] = value
            
            all_match_rows.append(entry)

df = pd.DataFrame(all_match_rows)
print(f"Done! Created a dataset with {len(df)} total match-performance rows.")
df.sample(5)

Done! Created a dataset with 26127 total match-performance rows.


,playerId,name,age,birthdate,status,origin_team,origin_comp,origin_season,origin_median,origin_matches,...,LOST_GROUND_DUELS_IN_PITCH_POSITION_OPPONENT_BOX,ASSISTS_BY_ACTION_SAVE,ASSISTS_AT_PHASE_SECOND_BALL,BYPASSED_OPPONENTS_FROM_PACKING_ZONE_WR,BYPASSED_OPPONENTS_NUMBER_FROM_PACKING_ZONE_WR,BALL_LOSS_REMOVED_TEAMMATES_FROM_PACKING_ZONE_WL,BYPASSED_DEFENDERS_RECEIVING_TO_PITCH_POSITION_FIRST_THIRD,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_IN_LANE_RIGHT_WING,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_OPPONENT_BOX,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_IN_PITCH_POSITION_OPPONENT_BOX
3752,21494,Kostas Lamprou,34.0,1991-09-18,DROPPED,RKC Waalwijk,eredivisie,2020-2021,0.691707,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7995,49491,Elia Caprile,24.0,2001-08-25,STAYED,Cagliari Calcio,serie_a,2025-2026,0.820000,72,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6218,103658,Nicolas Kristof,26.0,1999-12-20,STAYED,SV 07 Elversberg,bundesliga_2,2024-2025,0.710515,92,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3845,19571,Joël Drommel,29.0,1996-11-16,DROPPED,FC Twente Enschede,eredivisie,2020-2021,0.691707,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23490,23263,Philipp Köhn,27.0,1998-04-02,STAYED,AS Monaco,ligue1,2023-2024,0.771221,58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
numeric_df = df.select_dtypes(include=['number'])
correlations = numeric_df.corr()['step'].sort_values(ascending=False)
significant_kpis = correlations[abs(correlations) > 0.1]
print(f"Out of 1,000 KPIs, only {len(significant_kpis)} are statistically relevant.")
print(significant_kpis)

Out of 1,000 KPIs, only 107 are statistically relevant.
NEUTRAL_PASSES_BY_ACTION_THROW_IN                                       1.00000
BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_OWN_BOX        1.00000
step                                                                    1.00000
BALL_LOSS_ADDED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_MIDDLE_THIRD    1.00000
BYPASSED_OPPONENTS_RECEIVING_FROM_PACKING_ZONE_CM                       1.00000
                                                                         ...   
SHOT_XG_AT_PHASE_SECOND_BALL                                           -0.99661
BYPASSED_DEFENDERS_RECEIVING_TO_PITCH_POSITION_MIDDLE_THIRD            -1.00000
BYPASSED_DEFENDERS_RECEIVING_AT_PHASE_IN_POSSESSION                    -1.00000
BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_BY_ACTION_HEADER                  -1.00000
SHOT_XG_IN_LANE_RIGHT_HALF_SPACE                                       -1.00000
Name: step, Length: 107, dtype: float64


In [ ]:
metadata = [
    'playerId', 'name', 'age', 'status', 'direction', 
    'origin_team', 'origin_comp', 'origin_median', 
    'current_team', 'current_comp', 'current_median', 'step'
]

numeric_df = df.select_dtypes(include=['number'])
correlations = numeric_df.corr()['step'].abs()

useful_kpis = correlations[correlations > 0.1].index.tolist()

final_columns = list(set(metadata + useful_kpis))
useful_df = df[final_columns]

useful_df.to_csv('useful_keeper_dataset.csv', index=False)

print(f"Original columns: {df.shape[1]}")
print(f"Useful columns: {useful_df.shape[1]}")

Original columns: 1026
Useful columns: 117


In [39]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# 1. Define exactly which columns are METADATA (to be excluded)
metadata_to_drop = [
    'playerId', 'name', 'age', 'birthdate', 'status', 'direction',
    'origin_team', 'origin_comp', 'origin_season', 'origin_median',
    'origin_matches', 'current_team', 'current_comp', 'current_season',
    'current_median', 'current_matches', 'origin_match_dirs', 'current_match_dirs'
]

# 2. Isolate the KPIs (X) and the Result (y)
# We drop the metadata and the target ('step') from the inputs
X = useful_df.drop(columns=[c for c in metadata_to_drop if c in useful_df.columns] + ['step'])
y = useful_df['step']

# 3. Fill gaps (Keepers often have 'NaN' for rare stats like 'Penalty Saves')
X_clean = X.fillna(0)

from sklearn.ensemble import RandomForestRegressor

# 1. Increase 'n_estimators' to 1000 (More trees = deeper look)
# 2. Set 'max_features' to 'log2' or a specific number (Forces it to look at many different KPI combinations)
# 3. Use 'n_jobs=-1' to use all your CPU cores (It will be heavy, but faster)

heavy_model = RandomForestRegressor(
    n_estimators=1000, 
    max_depth=None, 
    min_samples_split=2,
    max_features='sqrt', # Important for 1000+ KPIs
    bootstrap=True,
    n_jobs=-1, 
    random_state=42
)

# Fit the model
heavy_model.fit(X_clean, y)

# 5. Extract the "Pure" Feature Importance
importance_df = pd.DataFrame({
    'Feature_KPI': X.columns,
    'Importance_Score': heavy_model.feature_importances_
}).sort_values(by='Importance_Score', ascending=False)

print("Top 10 Performance KPIs that actually trigger a league jump:")
print(importance_df.head(10))

Top 10 Performance KPIs that actually trigger a league jump:
                                          Feature_KPI  Importance_Score
59  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_AT_PHASE...          0.081636
12  BALL_WIN_REMOVED_OPPONENTS_IN_PITCH_POSITION_F...          0.058656
84  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_BY_ACTIO...          0.056854
5   BALL_WIN_REMOVED_OPPONENTS_BY_ACTION_INTERCEPTION          0.051305
69                BYPASSED_DEFENDERS_BY_ACTION_HEADER          0.046077
9                                        DEF_PXT_PASS          0.040054
74  BALL_WIN_ADDED_TEAMMATES_DEFENDERS_FROM_OPPONE...          0.034915
87  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_FROM_LAN...          0.033231
36  BYPASSED_OPPONENTS_RECEIVING_AT_PHASE_IN_POSSE...          0.028627
63  BYPASSED_OPPONENTS_RECEIVING_FROM_LANE_LEFT_HA...          0.027914


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# 1. Prepare Data (Strictly Performance KPIs only)
metadata_to_drop = [
    'playerId', 'name', 'age', 'birthdate', 'status', 'direction',
    'origin_team', 'origin_comp', 'origin_season', 'origin_median',
    'origin_matches', 'current_team', 'current_comp', 'current_season',
    'current_median', 'current_matches', 'origin_match_dirs', 'current_match_dirs'
]

X = df.drop(columns=[c for c in metadata_to_drop if c in df.columns] + ['step'])
y = df['step']
X_clean = X.fillna(0)

# 2. Split data (Testing on 'unseen' data is crucial for this test)
X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.2, random_state=42)

# 3. Fit the Model
print("Training the base model...")
model = RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 4. Run Permutation Importance (The Heavy Part)
# n_repeats=10 means it shuffles each KPI 10 different times to be sure
print("Running Permutation Test (this may take a few minutes)...")
result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

# 5. Extract and Visualize the Top 20
sorted_idx = result.importances_mean.argsort()[-20:]

plt.figure(figsize=(10, 8))
plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X_test.columns[sorted_idx])
plt.title("Permutation Importance (Top 20 KPIs)")
plt.xlabel("Decrease in $R^2$ Score when KPI is shuffled")
plt.tight_layout()
plt.savefig('permutation_importance_results.png')

# Print the ranking
importance_df = pd.DataFrame({
    'KPI': X_test.columns[sorted_idx],
    'Importance_Drop': result.importances_mean[sorted_idx]
}).sort_values(by='Importance_Drop', ascending=False)

print(importance_df)

Training the base model...
Running Permutation Test (this may take a few minutes)...


/home/daan/OHL-AI-Week/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/daan/OHL-AI-Week/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/daan/OHL-AI-Week/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/daan/OHL-AI-Week/.venv/lib/python3.14/site-packag

KeyboardInterrupt: 